In [4]:
!sudo update-alternatives --config python3


There are 2 choices for the alternative python3 (providing /usr/bin/python3).

  Selection    Path                Priority   Status
------------------------------------------------------------
* 0            /usr/bin/python3.7   2         auto mode
  1            /usr/bin/python3.6   1         manual mode
  2            /usr/bin/python3.7   2         manual mode

Press <enter> to keep the current choice[*], or type selection number: 1
update-alternatives: using /usr/bin/python3.6 to provide /usr/bin/python3 (python3) in manual mode


In [9]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd

from sklearn.metrics import classification_report
import time


import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [8]:
!pip install transformers

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 5, in <module>
    from pip._internal.cli.main import main
ModuleNotFoundError: No module named 'pip'


In [10]:
from transformers import pipeline

In [11]:
from google.colab import drive


In [12]:
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!ls "/content/drive/My Drive"

amazon_reviews_us_Electronics_v1_00.tsv
crawl-300d-2M.vec
S-Model11vModel12-Mult-10K_COLAB0-RERUN.ipynb
wk15_Replication.ipynb


In [14]:
#Electronics Dataset:

file = 'drive/My Drive/amazon_reviews_us_Electronics_v1_00.tsv'
df=pd.read_csv(file, sep="\t", header=0, on_bad_lines='skip')
df=df.dropna(subset=['review_headline', 'review_body', 'star_rating'])

In [15]:
from transformers import pipeline


In [16]:
#Model 12 (siebert) Helper Functions

In [18]:
def sentiment_classify(df_sample, column_name):
    sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")

    for i in range (0, len(df_sample[column_name])):

        text=df_sample[column_name][i]                 
        
        if len(text) > 514:
            text = text[:514]
            
        prediction={}
        prediction = sentiment_analysis(text)

        if prediction[0]['label']=='POSITIVE':
            df_sample.loc[i, ("sentiment_analysis")]=5

        elif prediction[0]['label']== 'NEGATIVE':
            df_sample.loc[i, ("sentiment_analysis")]=1
        else: 
            print("Error.")
            
      #  if i%1000==0:
      #     print ("\n sentiment_classify:   We are at i=", str(i))    
            
            
    return df_sample
            

In [19]:
#Model 11 / Ref cell 43 at https://github.com/sophiej-s/MSThesis-I/blob/main/wk15.ipynb
#Model 11  Helper Functions

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
lemma = WordNetLemmatizer()
ps = PorterStemmer()
import re


from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix





[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [21]:
from transformers import pipeline
def emotion_roberta(df_sample, column_name):
    classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)


    for i in range (0, len(df_sample[column_name])):

        #processed text
        text=df_sample[column_name][i]  

        if len(text) > 512:
            text = text[:512]

        prediction = classifier(text )

        for j in range(0,7):  #loop over the  emotions:
            df_sample.loc[i, ("roberta_"+column_name+prediction[0][j]['label'])]=prediction[0][j]['score'] #BP=body processed


       # if i%1000==0:
       #    print ("\n emotion_roberta:   We are at i=", str(i))    
            
            
    return df_sample

In [22]:
def text_process2(reviews, column_name):  #input is the dataframe
    for i  in range(0, reviews[column_name].count()):
       review_body=reviews.loc[i, (column_name)]  #tokens= word_tokenize(df_sample.loc[1, ('review_body')])
       review_body=re.sub('<br\s?\/>|<br>', " ", review_body)  #remove the br
       tokens= word_tokenize(review_body)
       tokens = [w.lower()  for w in tokens ]
       #tokens = [w for w in tokens if not w in stop_words]
       tokens = [w for w in tokens if w.isalpha()] #remove non alphabetic items like like 5 or ;
       tokens = [lemma.lemmatize(w) for w in tokens]
       # tokens = [ps.stem(w) for w in tokens]
       column_name_out=column_name+"_processed"
       reviews.loc[i, (column_name_out)]=' '.join(tokens)
       
      # if i%10000==0:
      #     print ("\n text_process:   We are at i=", str(i))
       
    return reviews

In [23]:

Roberta_Body=['roberta_review_bodyanger',
'roberta_review_bodydisgust',
'roberta_review_bodyfear',
'roberta_review_bodyjoy',
'roberta_review_bodyneutral',
'roberta_review_bodysadness',
'roberta_review_bodysurprise']



Roberta_Head=[
'roberta_review_headlineanger', 
'roberta_review_headlinedisgust',
'roberta_review_headlinefear', 
'roberta_review_headlinejoy',
'roberta_review_headlineneutral', 
'roberta_review_headlinesadness',
'roberta_review_headlinesurprise']


Roberta_Head_Processed=[
'roberta_review_headline_processedanger', 
'roberta_review_headline_processeddisgust',
'roberta_review_headline_processedfear', 
'roberta_review_headline_processedjoy',
'roberta_review_headline_processedneutral', 
'roberta_review_headline_processedsadness',
'roberta_review_headline_processedsurprise'] 

In [29]:
def run_SVC(input_df,input_y):
    target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive


    random_state=random.randint(0, 10000)
    print("SVC random state",random_state )

    X_train, X_test, y_train, y_test = train_test_split(input_df, input_y['star_rating'], test_size=0.33, random_state=random_state)
    
    #{'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}

    clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)
    
    return y_pred,y_test

    #clf.score(X_test, y_test)
    #y_test.value_counts()
   # print(clf.score(X_test, y_test))

#    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

#    print(confusion_matrix(y_test, y_pred))

In [25]:
 
def df2emd(word2vec_model, N_rewiews, column_name):
    word2vec_model_embeddings = WordVecVectorizer(word2vec_model)

    word2vec_model_embeddings_ave_one_review_list=[]
    embed_only=pd.DataFrame()
    
    for i in range(0,len(N_rewiews[column_name]) ) : 
    #for i in range(0,len(N_rewiews['review_body_process']) ) : 
        #list_words=[N_rewiews['review_body_process'][i]]
        list_words=[N_rewiews[column_name][i]]

        list_words=check_against_word2vec_model(list_words, word2vec_model)
        word2vec_embeddings_one_review=word2vec_model_embeddings.transform(list_words)
        word2vec_model_embeddings_ave_one_review_list.append(word2vec_embeddings_one_review)

    embed_only=pd.DataFrame(np.concatenate(word2vec_model_embeddings_ave_one_review_list))
    
    return embed_only

In [26]:
class WordVecVectorizer(object):
    def __init__(self, word2vec_model):
        self.word2vec_model = word2vec_model
        self.dim = 300
    def transform(self, X):
        return np.array([
            np.mean([self.word2vec_model[w] for w in texts.split() if w in self.word2vec_model]
                    or [np.zeros(self.dim)], axis=0)
            for texts in X
        ])

def check_against_word2vec_model(list_topics, word2vec_model):
    for i  in range(0, len(list_topics) ):
       tokens= word_tokenize(list_topics[i])
       tokens = [w for w in tokens if w in word2vec_model.key_to_index ]
       list_topics[i]=' '.join(tokens)
       return list_topics

In [27]:
import gensim
file_embeddings_fast = 'drive/My Drive/crawl-300d-2M.vec'
word2vec_model_fast = gensim.models.KeyedVectors.load_word2vec_format(file_embeddings_fast) 
print(word2vec_model_fast.vector_size)

300


In [ ]:
#Running Designs 11 and 12 

In [32]:
#Sampling the dataset     n_samples=1000



import random
random.seed(a=12, version=2)


for i in range(0, 7):
    
    random_state=random.randint(0, 6700)
    print("random_state: ",random_state) #print random_state for reproducibility

    n_samples=1000

    
    N_rewiews=df.loc[df['star_rating'] == 1].sample(n_samples, replace=False, random_state=random_state)
    N_rewiew2=df.loc[df['star_rating'] == 5].sample(n_samples, replace=False, random_state=random_state)

    samplesize=n_samples*2
    N_rewiews=N_rewiews.append(N_rewiew2)

    N_rewiews=N_rewiews.reset_index()
    N_rewiews['star_rating'].value_counts()

    #===================================
    #Running Model 11

    t_START = time.time()
    N_rewiews=text_process2(N_rewiews,'review_headline')

    emotion_roberta(N_rewiews, 'review_body')

    emotion_roberta(N_rewiews, 'review_headline')

    emotion_roberta(N_rewiews, 'review_headline_processed')
    df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 

    embed_only_fast_B=df2emd(word2vec_model_fast, N_rewiews, "review_body")
    embed_only_fast_HP=df2emd(word2vec_model_fast, N_rewiews, "review_headline")

    embed_combined=embed_only_fast_B.join(embed_only_fast_HP, lsuffix='_caller', rsuffix='_other')

    #combine the embeddings with the emotions
    embed_emptions_combined=embed_combined.join(df_sample_all_test, lsuffix='_caller', rsuffix='_other')


    #{'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}
    y_pred, y_test=run_SVC(embed_emptions_combined,N_rewiews)

    elapsed = time.time() - t_START
    print("Design 11 elapsed Time is:  ", elapsed)


    #Evaluation step:
    target_names = ['0 = rating of 1',  '1 = rating of 5'] 
    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))
    print ("===================================")



    #===================================
    #Running Model 12 (siebert)
    t_START = time.time()
    sentiment_classify(N_rewiews, 'review_body')
    elapsed = time.time() - t_START
    print("Design 12 elapsed Time is:  ", elapsed)

    #Evaluation step:
    y_pred= N_rewiews['sentiment_analysis']
    y_test=N_rewiews['star_rating']

    target_names = ['0 = rating of 1',  '1 = rating of 5'] 
    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))
    print ("===================================")

    #===================================







random_state:  3887
SVC random state 4407
Design 11 elapsed Time is:   448.3852951526642
                 precision    recall  f1-score   support

0 = rating of 1   0.954155  0.932773  0.943343       357
1 = rating of 5   0.922830  0.947195  0.934853       303

       accuracy                       0.939394       660
      macro avg   0.938492  0.939984  0.939098       660
   weighted avg   0.939774  0.939394  0.939445       660

Design 12 elapsed Time is:   1847.1883311271667
                 precision    recall  f1-score   support

0 = rating of 1   0.970149  0.975000  0.972569      1000
1 = rating of 5   0.974874  0.970000  0.972431      1000

       accuracy                       0.972500      2000
      macro avg   0.972512  0.972500  0.972500      2000
   weighted avg   0.972512  0.972500  0.972500      2000

random_state:  5386


/usr/local/lib/python3.7/dist-packages/transformers/pipelines/text_classification.py:90: UserWarning: `return_all_scores` is now deprecated, use `top_k=1` if you want similar functionnality
  "`return_all_scores` is now deprecated, use `top_k=1` if you want similar functionnality", UserWarning


SVC random state 8669
Design 11 elapsed Time is:   448.4597361087799
                 precision    recall  f1-score   support

0 = rating of 1   0.938462  0.927052  0.932722       329
1 = rating of 5   0.928358  0.939577  0.933934       331

       accuracy                       0.933333       660
      macro avg   0.933410  0.933314  0.933328       660
   weighted avg   0.933395  0.933333  0.933330       660

Design 12 elapsed Time is:   1842.2950296401978
                 precision    recall  f1-score   support

0 = rating of 1   0.976838  0.970000  0.973407      1000
1 = rating of 5   0.970209  0.977000  0.973592      1000

       accuracy                       0.973500      2000
      macro avg   0.973523  0.973500  0.973500      2000
   weighted avg   0.973523  0.973500  0.973500      2000

random_state:  5459


/usr/local/lib/python3.7/dist-packages/transformers/pipelines/text_classification.py:90: UserWarning: `return_all_scores` is now deprecated, use `top_k=1` if you want similar functionnality
  "`return_all_scores` is now deprecated, use `top_k=1` if you want similar functionnality", UserWarning


SVC random state 5730
Design 11 elapsed Time is:   446.0543713569641
                 precision    recall  f1-score   support

0 = rating of 1   0.916418  0.935976  0.926094       328
1 = rating of 5   0.935385  0.915663  0.925419       332

       accuracy                       0.925758       660
      macro avg   0.925901  0.925819  0.925756       660
   weighted avg   0.925959  0.925758  0.925754       660

Design 12 elapsed Time is:   1854.908546924591
                 precision    recall  f1-score   support

0 = rating of 1   0.976815  0.969000  0.972892      1000
1 = rating of 5   0.969246  0.977000  0.973108      1000

       accuracy                       0.973000      2000
      macro avg   0.973030  0.973000  0.973000      2000
   weighted avg   0.973030  0.973000  0.973000      2000

random_state:  1168


...

SVC random state 6139
Design 11 elapsed Time is:   457.633731842041
                 precision    recall  f1-score   support

0 = rating of 1   0.925414  0.954416  0.939691       351
1 = rating of 5   0.946309  0.912621  0.929160       309

       accuracy                       0.934848       660
      macro avg   0.935862  0.933519  0.934426       660
   weighted avg   0.935197  0.934848  0.934761       660

Design 12 elapsed Time is:   1862.614446401596
                 precision    recall  f1-score   support

0 = rating of 1   0.979508  0.956000  0.967611      1000
1 = rating of 5   0.957031  0.980000  0.968379      1000

       accuracy                       0.968000      2000
      macro avg   0.968270  0.968000  0.967995      2000
   weighted avg   0.968270  0.968000  0.967995      2000

random_state:  3952


/usr/local/lib/python3.7/dist-packages/transformers/pipelines/text_classification.py:90: UserWarning: `return_all_scores` is now deprecated, use `top_k=1` if you want similar functionnality
  "`return_all_scores` is now deprecated, use `top_k=1` if you want similar functionnality", UserWarning


SVC random state 4490
Design 11 elapsed Time is:   455.0089979171753
                 precision    recall  f1-score   support

0 = rating of 1   0.950292  0.931232  0.940666       349
1 = rating of 5   0.924528  0.945338  0.934817       311

       accuracy                       0.937879       660
      macro avg   0.937410  0.938285  0.937741       660
   weighted avg   0.938152  0.937879  0.937910       660

Design 12 elapsed Time is:   1870.5757858753204
                 precision    recall  f1-score   support

0 = rating of 1   0.981707  0.966000  0.973790      1000
1 = rating of 5   0.966535  0.982000  0.974206      1000

       accuracy                       0.974000      2000
      macro avg   0.974121  0.974000  0.973998      2000
   weighted avg   0.974121  0.974000  0.973998      2000

random_state:  5270


/usr/local/lib/python3.7/dist-packages/transformers/pipelines/text_classification.py:90: UserWarning: `return_all_scores` is now deprecated, use `top_k=1` if you want similar functionnality
  "`return_all_scores` is now deprecated, use `top_k=1` if you want similar functionnality", UserWarning


SVC random state 7540
Design 11 elapsed Time is:   461.0004301071167
                 precision    recall  f1-score   support

0 = rating of 1   0.950000  0.933526  0.941691       346
1 = rating of 5   0.928125  0.945860  0.936909       314

       accuracy                       0.939394       660
      macro avg   0.939062  0.939693  0.939300       660
   weighted avg   0.939593  0.939394  0.939416       660

Design 12 elapsed Time is:   1888.300594329834
                 precision    recall  f1-score   support

0 = rating of 1   0.970942  0.969000  0.969970      1000
1 = rating of 5   0.969062  0.971000  0.970030      1000

       accuracy                       0.970000      2000
      macro avg   0.970002  0.970000  0.970000      2000
   weighted avg   0.970002  0.970000  0.970000      2000

